In [6]:
from weights import WeightMatrix
from topologies import square_torus
import numpy as np
import warnings
warnings.filterwarnings("error")


In [58]:
def step_simulation(R, V, I, t=0, Delta=10, Eta=10, J=15, dt=1e-3):
    # Heun (RK2) instead of Euler + finite guards
    fR = Delta/np.pi + 2*R*V
    fV = V**2 + Eta + J*R + I - (np.pi**2)*(R**2)  # NOTE: +J*R (match your LaTeX)
    R1 = R + dt*fR
    V1 = V + dt*fV
    fR1 = Delta/np.pi + 2*R1*V1
    fV1 = V1**2 + Eta + J*R1 + I - (np.pi**2)*(R1**2)
    dR = 0.5*dt*(fR + fR1)
    np.nan_to_num(dR, copy=False, nan=0.0, posinf=1e6, neginf=-1e6)
    dR = np.clip(dR, -R/10, 1)
    R += dR
    R[:] = np.clip(R, 1e-5, 1e+2)
    dV = 0.5*dt*(fV + fV1)
    np.nan_to_num(dV, copy=False, nan=0.0, posinf=1e6, neginf=-1e6)
    dV = np.clip(dV, -1, 1)
    V += dV
    V[:] = np.clip(V, -1e2, 5)

    np.nan_to_num(R, copy=False, nan=0.0, posinf=1e6, neginf=-1e6)
    np.nan_to_num(V, copy=False, nan=0.0, posinf=1e6, neginf=-1e6)
    return t + dt, dR


In [ ]:
import threading, queue
import ipywidgets as w

t = 0
weights = WeightMatrix(square_torus(20),
                       weight_initializer=lambda **s: np.exp(np.random.normal(size=s['size'], loc=.1, scale=.25)))
children = np.array([*weights.network.values()])

R, V = np.zeros((2, weights.size))
V.fill(-.1)
R.fill(.1)
I = np.zeros(weights.size)
weights.check = True


def update_I(I, V, weights, t):
    I.fill(0)

    I[[0, -1]] = 3 * np.sin(np.pi * t / 20)
    children = np.array([*weights.network.values()])
    delta = V[:,None] * weights[np.arange(weights.size)[:, None], children]
    np.add.at(I, children.ravel(), delta.ravel())
    I[[0, -1]] = 3 * np.sin(np.pi * t / 20)
    I += np.random.normal(size=weights.size, loc=.01, scale=0.05)


def update_W(weights, R, dR):
    i = np.arange(weights.size)[:, None]
    j = np.array([*weights.network.values()])
    weights.at[i, j] <<= (R[:, None] * dR[j] - R[j] * dR[:, None])


import plotly.graph_objects as go, time

# initial setup
fig = go.FigureWidget()
fig.update_layout(width=500, height=500, margin=dict(l=0, r=0, b=0, t=0))

heat = fig.add_heatmap(z=np.zeros((20, 20)), colorscale="Viridis", zmax=10, zmin=0)

# make the figure a square
display(fig)
minvs = []
maxvs = []

# frame interval slider (interactive)
frame_interval = w.IntSlider(value=5, min=1, max=100, step=1, description="Frame N", continuous_update=True)

# async display thread
viz_q = queue.Queue(maxsize=2)
stop_flag = threading.Event()

# double-buffer latest frame (lock-protected)
latest_frame = {"data": None, "t": -float("inf")}
latest_lock = threading.Lock()


def viz_loop():
    last_drawn_t = -float("inf")
    while not stop_flag.is_set():
        try:
            # Always consume to the newest frame to minimize latency
            item = viz_q.get(timeout=0.02)
            while True:
                try:
                    item = viz_q.get_nowait()
                except queue.Empty:
                    break
            with latest_lock:
                latest_frame["data"], latest_frame["t"] = item
        except queue.Empty:
            pass

        with latest_lock:
            mP_grid = latest_frame["data"]
            cur_t = latest_frame["t"]

        if mP_grid is not None and cur_t >= last_drawn_t:
            with fig.batch_update():
                fig.data[0].z = mP_grid
                # Avoid relayout cost each frame; title only when significantly changed
            last_drawn_t = cur_t

        # tiny sleep to yield to UI thread
        time.sleep(0.01)


viz_thread = threading.Thread(target=viz_loop, daemon=True, name="viz_thread")
viz_thread.start()
display(frame_interval)

# simulation loop
try:
    for tick in range(100000):
        update_I(I, V, weights, t)
        t, dR = step_simulation(R, V, I, t, dt=1e-2)
        update_W(weights, R, dR/10)

        N = max(1, int(frame_interval.value))
        if tick % N == 0:
            mP_grid = R.reshape(20, 20)
            # Avoid extra copy; let viz thread overwrite on next frame
            try:
                viz_q.put_nowait((mP_grid, t))
            except queue.Full:
                # Drop oldest then enqueue latest to reduce latency
                try:
                    _ = viz_q.get_nowait()
                except queue.Empty:
                    pass
                finally:
                    try:
                        viz_q.put_nowait((mP_grid, t))
                    except queue.Full:
                        pass
finally:
    stop_flag.set()
    viz_thread.join(timeout=1)


FigureWidget({
    'data': [{'colorscale': [[0.0, '#440154'], [0.1111111111111111, '#482878'],
                             [0.2222222222222222, '#3e4989'], [0.3333333333333333,
                             '#31688e'], [0.4444444444444444, '#26828e'],
                             [0.5555555555555556, '#1f9e89'], [0.6666666666666666,
                             '#35b779'], [0.7777777777777778, '#6ece58'],
                             [0.8888888888888888, '#b5de2b'], [1.0, '#fde725']],
              'type': 'heatmap',
              'uid': '78d1d1ff-38f0-4e98-ac96-8bb74bb253e5',
              'z': {'bdata': ('AAAAAAAAAAAAAAAAAAAAAAAAAAAAAA' ... 'AAAAAAAAAAAAAAAAAAAAAAAAAAAAA='),
                    'dtype': 'f8',
                    'shape': '20, 20'},
              'zmax': 10,
              'zmin': 0}],
    'layout': {'height': 500, 'margin': {'b': 0, 'l': 0, 'r': 0, 't': 0}, 'template': '...', 'width': 500}
})

IntSlider(value=5, description='Frame N', min=1)

In [52]:
weights[1,2]

36986.641584164005

In [93]:
# set the IOPub message limit much higher
import os, json, sys

# Increase limits for the running IPython kernel process (best effort)
os.environ["IPYKERNEL_CELL_NAME"] = "high_iopub_limits"
try:
    from IPython import get_ipython

    ip = get_ipython()
    if ip is not None and hasattr(ip, "kernel") and hasattr(ip.kernel, "session"):
        # These config keys are read at startup; for a running kernel we adjust traitlets directly if present
        if hasattr(ip.kernel, "iopub_thread") and hasattr(ip.kernel.iopub_thread, "rate_limit"):
            # Disable rate limiting by setting huge limits
            ip.kernel.iopub_thread.rate_limit = 1e10
            ip.kernel.iopub_thread.max_msg_rate = 1e9
            ip.kernel.iopub_thread.max_msg_size = int(1e9)
except Exception as e:
    print("Could not adjust IOPub limits at runtime:", e, file=sys.stderr)

# Also tell Jupyter Server (if it respects env for spawned kernels later in this session)
os.environ["IPKernelApp.iopub_msg_rate_limit"] = "1000000000"
os.environ["IPKernelApp.iopub_data_rate_limit"] = "1.0e11"


In [90]:
ip.kernel.iopub_thread.rate_limit = 1e10

In [89]:
os.environ["IPKernelApp.iopub_msg_rate_limit"] = "1000000000"

In [4]:
import numpy as np, time
import plotly.graph_objects as go

fig = go.FigureWidget([go.Scatter(x=[], y=[], mode="lines")])
display(fig)

y = []
for t in range(1000):
    y.append(np.sin(t/10))
    with fig.batch_update():
        fig.data[0].x = np.arange(len(y))
        fig.data[0].y = y
    time.sleep(0.001)


FigureWidget({
    'data': [{'mode': 'lines', 'type': 'scatter', 'uid': 'eaec5825-aaf4-4163-9baf-5f82664efb24', 'x': [], 'y': []}],
    'layout': {'template': '...'}
})

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [5]:
 import ipywidgets as w, plotly.io as pio
print("ipywidgets", w.__version__)
_ = w.IntSlider()  # should render a slider


ipywidgets 8.1.5
